# Step 8: Which places lose their road to health care?

Combines the flood maps (Step 6) with the road network and health facilities (Step 7).

**Run in Google Colab.** No GPU. About 15 to 30 minutes.

**Before running, put two folders in your Google Drive (My Drive):**
- `cutoff_flood_run1`: the files from Step 6's `outputs/` folder (at least `flood_bits.tif` and `flood_count.tif`)
- `cutoff_network`: Tanush's folder (if it was shared with you: right-click it in *Shared with me* > *Organize* > *Add shortcut* > *My Drive*)

**Method**
1. **Population units.** People are counted in fixed 500 m squares built straight from WorldPop, not from building clusters (see NOTES.md for why). Each square is placed at its population-weighted centre and joined to the nearest road junction that can reach a health facility on a normal day, up to 500 m away.
2. **Blocked roads.** Points are sampled every 5 m along and across each road's 20 m strip. For each of the 5 models, a road stretch is blocked if more than 20% of its points are flooded (also 10% and 40%). Bridges are never blocked.
3. **Routes.** Shortest road distance from every junction to the nearest health facility, on the normal network and on each flooded network.
4. **Status per square and model:** cut off (no route), long detour (at least 2 times the normal route), or reachable.
5. **Likelihood.** The share of the 5 models in which a square is cut off: 0%, 20%, ..., 100%.

**Not counted as flood-cut-off:** squares more than 500 m from any road that reaches a facility on a normal day. They are reported separately as "no road access", because their problem is the map or the place, not the flood.

In [ ]:
!pip install -q osmnx > /dev/null

In [ ]:
import os, sys, subprocess, json, time
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
REPO = os.environ.get("CUTOFF_REPO", "/content/cutoff-flood")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=False)
sys.path.insert(0, REPO)
from src import config

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = os.environ.get("CUTOFF_BASE", ".")
FLOOD_DIR = f"{BASE}/cutoff_flood_run1"
NET_DIR = f"{BASE}/cutoff_network"
OUT = os.environ.get("CUTOFF_OUT", "/content/outputs_cutoff" if ON_COLAB else "./outputs_cutoff")
RAW = os.environ.get("CUTOFF_RAW", "/content/raw" if ON_COLAB else "./raw")
for d in (OUT, f"{OUT}/figures", RAW):
    os.makedirs(d, exist_ok=True)
for p in (f"{FLOOD_DIR}/flood_bits.tif", f"{NET_DIR}/network.gpkg", f"{NET_DIR}/facilities.gpkg", f"{NET_DIR}/districts.gpkg"):
    assert os.path.exists(p), f"Missing {p}. See the instructions at the top."
print("Inputs found")

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from scipy import sparse
from scipy.sparse.csgraph import dijkstra
from scipy.spatial import cKDTree

CRS = config.WORK_CRS
L, B, R, T = config.STUDY_BBOX
N_MODELS = config.ENSEMBLE_SIZE
THRESHOLDS = config.ROAD_BLOCKED_SENSITIVITY     # [0.10, 0.20, 0.40]
MAIN_THR = config.ROAD_BLOCKED_SHARE             # 0.20

## 1. Load the network, facilities and districts

In [ ]:
t0 = time.time()
edges = gpd.read_file(f"{NET_DIR}/network.gpkg", layer="edges",
                      columns=["edge_id", "u", "v", "cls", "is_bridge", "length"], read_geometry=False)
nodes = gpd.read_file(f"{NET_DIR}/network.gpkg", layer="nodes", columns=["osmid", "x", "y"], read_geometry=False)
districts = gpd.read_file(f"{NET_DIR}/districts.gpkg").to_crs(CRS)
fac = gpd.read_file(f"{NET_DIR}/facilities.gpkg").to_crs(CRS)

# index every junction 0..n-1 for fast routing
nodes = nodes.drop_duplicates("osmid").reset_index(drop=True)
idx = pd.Series(np.arange(len(nodes)), index=nodes.osmid.values)
edges["iu"] = idx.reindex(edges.u.values).values
edges["iv"] = idx.reindex(edges.v.values).values
edges = edges.dropna(subset=["iu", "iv"]).astype({"iu": int, "iv": int})
edges["is_bridge"] = edges.is_bridge.astype(bool)
NX, NY = nodes.x.values, nodes.y.values
print(f"{len(nodes):,} junctions, {len(edges):,} road stretches ({time.time() - t0:.0f} s)")

# keep only places to go for care
name = fac["name"].fillna("").str.lower()
bad_name = name.str.contains("|".join(config.FACILITY_EXCLUDE_NAME_WORDS))
fac["counted"] = fac.kind.isin(config.FACILITY_KINDS) & ~bad_name & fac.on_network.astype(bool)
fac["inode"] = idx.reindex(fac.node.values).values
print(f"Facilities: {len(fac)} in file, {fac.counted.sum()} counted "
      f"(dropped {(~fac.kind.isin(config.FACILITY_KINDS)).sum()} by type, {bad_name.sum()} by name, "
      f"{(~fac.on_network.astype(bool)).sum()} too far from a road)")
fac = fac[fac.counted & fac.inode.notna()].copy()
fac["inode"] = fac.inode.astype(int)

## 2. Routing helpers

In [ ]:
def make_graph_min(blocked=None):
    e = edges if blocked is None else edges[~blocked]
    a, b = np.minimum(e.iu.values, e.iv.values), np.maximum(e.iu.values, e.iv.values)
    df = pd.DataFrame({"a": a, "b": b, "w": np.maximum(e.length.values, 0.01)}).groupby(["a", "b"]).w.min().reset_index()
    m = sparse.coo_matrix((np.r_[df.w, df.w], (np.r_[df.a, df.b], np.r_[df.b, df.a])), shape=(len(nodes), len(nodes)))
    return m.tocsr()

def dist_to_care(blocked=None):
    """Road distance (m) from every junction to the nearest counted facility."""
    return dijkstra(make_graph_min(blocked), directed=False, indices=np.unique(fac.inode.values), min_only=True)

t0 = time.time()
dry = dist_to_care()
print(f"Normal day: {np.isfinite(dry).mean():.1%} of junctions can reach a facility ({time.time() - t0:.0f} s)")

## 3. Population units: 500 m squares from WorldPop

In [ ]:
from rasterio.windows import from_bounds
from pyproj import Transformer

WORLDPOP_URL = ("https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/"
                "2020/BSGM/IND/ind_ppp_2020_constrained.tif")
WORLDPOP = f"{RAW}/worldpop_ind_2020_constrained.tif"
if not os.path.exists(WORLDPOP) or os.path.getsize(WORLDPOP) < 5e8:
    subprocess.run(["wget", "-q", "-O", WORLDPOP, WORLDPOP_URL], check=True)

with rasterio.open(WORLDPOP) as src:
    win = from_bounds(L, B, R, T, src.transform).round_offsets().round_lengths()
    pop = src.read(1, window=win).astype(np.float64)
    tr = src.window_transform(win)
    nod = src.nodata
pop[(pop < 0) | (pop == nod) | ~np.isfinite(pop)] = 0
rr, cc = np.nonzero(pop > 0)
lon, lat = tr * (cc + 0.5, rr + 0.5)
px, py = Transformer.from_crs("EPSG:4326", CRS, always_xy=True).transform(lon, lat)
people = pop[rr, cc]

S = config.POP_UNIT_M
cell = pd.DataFrame({"gx": np.floor(px / S).astype(int), "gy": np.floor(py / S).astype(int),
                     "p": people, "px": px * people, "py": py * people})
u = cell.groupby(["gx", "gy"]).sum().reset_index()
u["x"], u["y"] = u.px / u.p, u.py / u.p
u = u[u.p >= config.MIN_UNIT_PEOPLE]
units = gpd.GeoDataFrame({"people": u.p.round(0).values, "gx": u.gx.values, "gy": u.gy.values},
                         geometry=gpd.points_from_xy(u.x, u.y), crs=CRS)
units = gpd.sjoin(units, districts[["district", "geometry"]], predicate="within").drop(columns="index_right")
units = units.reset_index(drop=True)
units["unit_id"] = np.arange(len(units))
print(f"{len(units):,} squares with people in the three districts, {units.people.sum():,.0f} people")
print(units.groupby("district").people.sum().round(0))

## 4. Join each square to the road network

Each square joins the nearest junction **that can reach a facility on a normal day**, up to 500 m away. This skips small disconnected bits of mapped road. Squares with no such junction within 500 m are "no road access". If the nearest junction of any kind is close but disconnected, the square is flagged as a likely map gap.

In [ ]:
reach = np.isfinite(dry)
tree_ok = cKDTree(np.c_[NX[reach], NY[reach]])
tree_all = cKDTree(np.c_[NX, NY])
ok_ids = np.nonzero(reach)[0]
ux, uy = units.geometry.x.values, units.geometry.y.values
d_ok, j_ok = tree_ok.query(np.c_[ux, uy])
d_all, _ = tree_all.query(np.c_[ux, uy])
units["inode"] = ok_ids[j_ok]
units["snap_m"] = d_ok.round(1)
units["no_road_access"] = d_ok > config.SNAP_MAX_M
units["map_gap"] = units.no_road_access & (d_all <= config.SNAP_MAX_M)
units["normal_route_m"] = np.where(units.no_road_access, np.nan, dry[units.inode] + units.snap_m)
a = units[~units.no_road_access]
print(f"No road access: {units.no_road_access.sum():,} squares, {units.loc[units.no_road_access, 'people'].sum():,.0f} people "
      f"({units.loc[units.no_road_access, 'people'].sum() / units.people.sum():.1%}); "
      f"of these, likely map gaps: {units.map_gap.sum():,}")
print(f"Normal route to care: median {a.normal_route_m.median() / 1000:.1f} km, 90th percentile {a.normal_route_m.quantile(.9) / 1000:.1f} km")

## 5. Which road stretches are flooded?

For each stretch, points every 5 m along it and at 4 positions across its 20 m strip are checked against each model's flood map. The flooded share per model is the fraction of those points that are flooded.

In [ ]:
with rasterio.open(f"{FLOOD_DIR}/flood_bits.tif") as f:
    bits = f.read(1)
    ftr = f.transform
H, W = bits.shape
inv = ~ftr

def flooded_share(e, bits_img, chunk=50_000):
    """(len(e), N_MODELS) array: share of each stretch's strip points flooded, per model."""
    step = config.STRIP_SAMPLE_M
    half = config.ROAD_BUFFER_M / 2
    offsets = np.arange(-half + step / 2, half, step)      # e.g. -7.5, -2.5, 2.5, 7.5
    out = np.zeros((len(e), N_MODELS), dtype=np.float32)
    x0, y0 = NX[e.iu.values], NY[e.iu.values]
    x1, y1 = NX[e.iv.values], NY[e.iv.values]
    for s in range(0, len(e), chunk):
        sl = slice(s, s + chunk)
        ax, ay, bx, by = x0[sl], y0[sl], x1[sl], y1[sl]
        ln = np.hypot(bx - ax, by - ay)
        nx_, ny_ = np.where(ln > 0, -(by - ay) / np.maximum(ln, 1e-9), 0), np.where(ln > 0, (bx - ax) / np.maximum(ln, 1e-9), 0)
        n_along = np.maximum(np.ceil(ln / step).astype(int), 1) + 1
        eid = np.repeat(np.arange(len(ax)), n_along)
        start = np.repeat(np.cumsum(n_along) - n_along, n_along)
        t = (np.arange(len(eid)) - start) / np.maximum(np.repeat(n_along, n_along) - 1, 1)
        cnt = np.zeros(len(ax)); acc = np.zeros((len(ax), N_MODELS))
        for d in offsets:
            x = ax[eid] + t * (bx - ax)[eid] + d * nx_[eid]
            y = ay[eid] + t * (by - ay)[eid] + d * ny_[eid]
            col, row = inv * (x, y)
            row, col = np.floor(row).astype(int), np.floor(col).astype(int)
            inside = (row >= 0) & (row < H) & (col >= 0) & (col < W)
            v = np.zeros(len(x), dtype=np.uint8)
            v[inside] = bits_img[row[inside], col[inside]]
            for k in range(N_MODELS):
                acc[:, k] += np.bincount(eid, weights=(v >> k) & 1, minlength=len(ax))
            cnt += np.bincount(eid, minlength=len(ax))
        out[sl] = acc / cnt[:, None]
    return out

t0 = time.time()
share = flooded_share(edges, bits)
del bits
share[edges.is_bridge.values] = 0          # bridges are never blocked
km = edges.length.values / 1000
print(f"Done ({time.time() - t0:.0f} s)")
for thr in THRESHOLDS:
    blk = share > thr
    print(f"threshold {thr:.0%}: blocked road km per model = {[round(km[blk[:, k]].sum()) for k in range(N_MODELS)]}")

## 6. Route every square on every flooded network

5 models x 3 thresholds = 15 flooded networks, plus one network from the majority map (a stretch blocked if 3 or more models block it), which stands for "one flood map, no uncertainty".

In [ ]:
nodes_needed = units.inode.values
res = {}
t0 = time.time()
for thr in THRESHOLDS:
    blk = share > thr
    for k in range(N_MODELS):
        res[(thr, k)] = dist_to_care(blk[:, k])[nodes_needed]
    res[(thr, "majority")] = dist_to_care(blk.sum(1) >= 3)[nodes_needed]
    print(f"threshold {thr:.0%} done ({time.time() - t0:.0f} s)")

normal = dry[nodes_needed]
ok = ~units.no_road_access.values

def status(d):
    s = np.full(len(d), "reachable", dtype=object)
    s[np.isfinite(normal) & (d >= config.DETOUR_LONG_RATIO * np.maximum(normal, 1))] = "long detour"
    s[~np.isfinite(d)] = "cut off"
    s[~ok] = "no road access"
    return s

for thr in THRESHOLDS:
    tag = f"{int(thr * 100)}"
    cut = np.stack([~np.isfinite(res[(thr, k)]) & ok for k in range(N_MODELS)], axis=1)
    units[f"cutoff_models_{tag}"] = cut.sum(1)
    units[f"cutoff_likelihood_{tag}"] = cut.mean(1)
    units[f"majority_status_{tag}"] = status(res[(thr, "majority")])
    ratios = np.stack([np.where(np.isfinite(res[(thr, k)]), res[(thr, k)] / np.maximum(normal, 1), np.nan)
                       for k in range(N_MODELS)], axis=1)
    units[f"median_detour_ratio_{tag}"] = np.nanmedian(np.where(np.isfinite(ratios), ratios, np.nan), axis=1).round(2)
print("Done")

## 6b. Is the square itself flooded?

The project is about places that are **dry but cut off**. People whose own square is mostly under water are a different group (they need rescue, not a road). For each 500 m square we measure the share of its area that the majority of models (3 or more of 5) call flooded.

In [ ]:
with rasterio.open(f"{FLOOD_DIR}/flood_count.tif") as f:
    fc = f.read(1)
    ftr = f.transform
H, W = fc.shape
S = config.POP_UNIT_M
xs = ftr.c + (np.arange(W) + 0.5) * ftr.a
gx_col = np.floor(xs / S).astype(np.int64)
key_units = pd.Series(np.arange(len(units)), index=units.gx.astype(np.int64) * 10**7 + units.gy.astype(np.int64))
wet = np.zeros(len(units)); tot = np.zeros(len(units))
for r0 in range(0, H, 500):
    blk = fc[r0:r0 + 500]
    ys = ftr.f + (np.arange(r0, r0 + len(blk)) + 0.5) * ftr.e
    gy_row = np.floor(ys / S).astype(np.int64)
    keys = (gx_col[None, :] * 10**7 + gy_row[:, None]).ravel()
    pos = key_units.reindex(keys).values
    m = ~np.isnan(pos)
    pos = pos[m].astype(int)
    tot += np.bincount(pos, minlength=len(units))
    wet += np.bincount(pos, weights=(blk.ravel()[m] >= 3), minlength=len(units))
del fc
units["square_flooded_share"] = np.where(tot > 0, wet / np.maximum(tot, 1), 0).round(3)
units["square_mostly_dry"] = units.square_flooded_share < 0.25
print(f"People in squares at least 25% flooded (majority map): "
      f"{units.loc[~units.square_mostly_dry, 'people'].sum():,.0f}")

## 7. Results

**Main setting: a road is blocked if more than 20% of its strip is flooded.**

In [ ]:
def bucket(lk):
    return pd.cut(lk, [-0.01, 0.0, 0.4, 0.8, 1.0], labels=["0 of 5", "1 to 2 of 5", "3 to 4 of 5", "5 of 5"])

def connected(snap_max):
    return units.snap_m <= snap_max

def table(thr, snap_max=config.SNAP_MAX_M):
    tag = f"{int(thr * 100)}"
    a = units[connected(snap_max) & units.square_mostly_dry].copy()
    a["models_cut_off"] = bucket(a[f"cutoff_likelihood_{tag}"])
    t = a.pivot_table(index="district", columns="models_cut_off", values="people", aggfunc="sum", observed=False).fillna(0)
    t.loc["Total"] = t.sum()
    return t.round(0).astype(int)

print("People in MOSTLY DRY squares cut off from every health facility by road,")
print("by how many of the 5 models agree (road blocked if more than 20% of its strip is flooded)")
main = table(MAIN_THR)
main

In [ ]:
rows = []
for snap_max in (config.SNAP_MAX_M, 1000):
    con = connected(snap_max)
    for thr in THRESHOLDS:
        tag = f"{int(thr * 100)}"
        a = units[con & units.square_mostly_dry]
        maj = a[f"majority_status_{tag}"].where(a.snap_m <= config.SNAP_MAX_M,
                                                np.where(np.isfinite(res[(thr, "majority")][a.unit_id]), "reachable", "cut off"))
        lk = a[f"cutoff_likelihood_{tag}"] if snap_max == config.SNAP_MAX_M else pd.Series(
            np.stack([~np.isfinite(res[(thr, k)][a.unit_id]) for k in range(N_MODELS)], 1).mean(1), index=a.index)
        rows.append({
            "join to road within (m)": snap_max,
            "road blocked if flooded share >": f"{thr:.0%}",
            "people counted (dry, on network)": int(a.people.sum()),
            "cut off, majority map": int(a.people[maj == "cut off"].sum()),
            "cut off, all 5 models": int(a.people[lk == 1].sum()),
            "cut off, any model": int(a.people[lk > 0].sum()),
            "uncertain (1 to 4 of 5)": int(a.people[(lk > 0) & (lk < 1)].sum()),
        })
sens = pd.DataFrame(rows)
print(f"Not counted: no road within {config.SNAP_MAX_M} m on a normal day: "
      f"{units.loc[~connected(config.SNAP_MAX_M), 'people'].sum():,.0f} people; "
      f"own square at least 25% flooded: {units.loc[connected(config.SNAP_MAX_M) & ~units.square_mostly_dry, 'people'].sum():,.0f} people")
sens

In [ ]:
# Save
keep = [c for c in units.columns if c not in ("gx", "gy")]
units[keep].to_file(f"{OUT}/units_cutoff.gpkg", driver="GPKG")
main.to_csv(f"{OUT}/people_cutoff_by_agreement_20pct.csv")
sens.to_csv(f"{OUT}/sensitivity_thresholds.csv", index=False)
blocked_main = (share > MAIN_THR).sum(1)
pd.DataFrame({"edge_id": edges.edge_id.values, "models_blocked_20pct": blocked_main.astype(np.uint8),
              **{f"share_model{k}": share[:, k].round(3) for k in range(N_MODELS)}}).to_csv(f"{OUT}/edges_flooded.csv.gz", index=False)
with open(f"{OUT}/run_info.json", "w") as f:
    json.dump({"pop_unit_m": config.POP_UNIT_M, "snap_max_m": config.SNAP_MAX_M, "thresholds": THRESHOLDS,
               "main_threshold": MAIN_THR, "detour_long_ratio": config.DETOUR_LONG_RATIO,
               "facility_kinds": sorted(config.FACILITY_KINDS), "facilities_counted": int(len(fac)),
               "strip_width_m": config.ROAD_BUFFER_M, "strip_sample_m": config.STRIP_SAMPLE_M,
               "units": int(len(units)), "people": float(units.people.sum())}, f, indent=1)
print(sorted(os.listdir(OUT)))

## 8. Map

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(13, 10))
districts.boundary.plot(ax=ax, color="black", linewidth=0.8)
a = units[~units.no_road_access]
col = f"cutoff_models_{int(MAIN_THR * 100)}"
a[a[col] == 0].plot(ax=ax, color="#d9d9d9", markersize=2)
cmap = ListedColormap(["#fdd49e", "#fc8d59", "#e34a33", "#b30000", "#7f0000"])
hit = a[a[col] > 0]
hit.plot(ax=ax, column=col, cmap=cmap, vmin=0.5, vmax=5.5, markersize=8, legend=True,
         legend_kwds={"label": "models in which the square is cut off (of 5)", "shrink": 0.6})
units[units.no_road_access].plot(ax=ax, color="#3182bd", markersize=3)
fac.plot(ax=ax, color="green", marker="+", markersize=40)
ax.set_title("500 m squares cut off by road from all health facilities, 29 July 2026\n"
             "grey: reachable, blue: no road access on a normal day, green +: health facility")
ax.set_axis_off(); plt.tight_layout(); plt.savefig(f"{OUT}/figures/cutoff_map.png", dpi=120); plt.show()

In [ ]:
if ON_COLAB:
    import shutil
    shutil.copytree(OUT, f"{BASE}/cutoff_results_run1", dirs_exist_ok=True)
    print("Copied to My Drive/cutoff_results_run1")

## Read before quoting any number

- The flood map has **not yet been checked against hand-marked Assam tiles**. Until then, every number here is provisional.
- "Cut off" means **no road route** to any counted facility. People may still reach care by boat, on foot through shallow water, or by air rescue.
- The likelihood comes from 5 models trained on the same data, so it probably **understates** the real uncertainty.
- Missing roads in OpenStreetMap make places look more cut off; mud tracks mapped as roads make them look less cut off. The visual check in Step 7 measures this.